# GeoLatent — Interactive 3-D ML Visualisation Demo

**GeoLatent** is a geometry-aware, model-intelligent 3-D visualisation library
for machine learning workflows.  This notebook demonstrates:

1. **Decision geometry** — true 3-D decision boundary isosurfaces for a kernel SVM
2. **Gradient Boosting geometry** — multi-class probability surfaces
3. **Latent space inspection** — high-dimensional embedding structure via PCA / t-SNE
4. **Custom configuration** — theme overrides and projection options
5. **Advanced pipeline** — using the lower-level API for full compositional control

> All figures are fully interactive — rotate, zoom, and toggle traces in the legend.

## 0 · Installation

In [ ]:
"""
CELL 0 — Dependency installation
Run this cell only on environments that don't already have the packages
(e.g. fresh Colab, base conda). After running, restart the kernel.
"""
import sys, subprocess

def _run(args):
    subprocess.run([sys.executable, "-m", "pip"] + args, check=True)

print(f"Python: {sys.executable}")
print("\n[1/3] Upgrading NumPy, SciPy, Plotly ...")
_run(["install", "-q", "--upgrade", "numpy>=1.23", "scipy>=1.9", "plotly>=5.13"])

# Force-reinstall sklearn so its compiled extensions match the installed NumPy.
# This is necessary when conda's sklearn was compiled against a different NumPy ABI.
print("[2/3] Force-reinstalling scikit-learn ...")
_run(["install", "-q", "--force-reinstall", "--no-deps", "scikit-learn>=1.5"])

print("[3/3] Installing geolatent ...")
_run(["install", "-q", "geolatent"])

print("\nDone.  Restart the kernel now, then run from Cell 1 onwards.")

In [ ]:
import numpy as np
import plotly.io as pio
from sklearn.datasets import make_classification, make_blobs
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import geolatent as nv
from geolatent import (
    visualize_decision_geometry,
    inspect_latent_space,
    VisualizationConfig,
    DARK_SCIENTIFIC,
)

def _detect_renderer() -> str:
    try:
        import google.colab  # noqa: F401
        return "colab"
    except ImportError:
        pass
    return "notebook_connected"

pio.renderers.default = _detect_renderer()

print(f"GeoLatent v{nv.__version__} loaded.")
print(f"Plotly renderer: {pio.renderers.default}")

---
## 1 · Decision Geometry of a Radial-Basis SVM

We train a 3-class RBF-kernel SVM on a synthetic 20-dimensional classification
problem.  `visualize_decision_geometry` projects the data to 3 principal
components, builds a 30³ prediction mesh by inverse-transforming the PCA grid
back to the original 20-D feature space, evaluates the model at every vertex,
and renders probability isosurfaces at the P = 0.50 boundary (solid shells)
and P = 0.70 / P = 0.85 confidence layers (inner shells).

In [ ]:
# ── Synthetic 3-class, 20-feature dataset ────────────────────────────
X_svm, y_svm = make_classification(
    n_samples=450,
    n_features=20,
    n_classes=3,
    n_informative=10,
    n_redundant=4,
    n_clusters_per_class=1,
    random_state=42,
)

# ── Fit RBF-SVM with probability calibration ─────────────────────────
svm = SVC(kernel="rbf", C=10.0, gamma="scale", probability=True, random_state=42)
svm.fit(X_svm, y_svm)

print(f"Training accuracy: {svm.score(X_svm, y_svm):.3f}")

In [ ]:
fig_svm = visualize_decision_geometry(
    model=svm,
    X=X_svm,
    y=y_svm,
    projection_method="pca",
    mesh_resolution=30,
    show_surface=True,
    show_confidence=True,   # nested probability shells at 0.70 and 0.85
    show_scatter=True,
    show_centroids=True,
    show_ellipsoids=True,
    title="RBF-SVM Decision Geometry — 3-class, 20-D features",
    class_names={0: "Class Alpha", 1: "Class Beta", 2: "Class Gamma"},
)
fig_svm.show()

---
## 2 · Gradient Boosting — Multi-class Decision Manifolds

Gradient Boosting ensembles carve the feature space into complex, non-convex
regions.  The PCA projection captures the directions of maximum variance, so
the rendered isosurfaces reflect the actual model partitioning along the most
informative axes — not an arbitrary 2-D cross-section.

In [ ]:
X_gbm, y_gbm = make_classification(
    n_samples=600,
    n_features=30,
    n_classes=4,
    n_informative=12,
    n_redundant=6,
    n_clusters_per_class=2,
    random_state=7,
)

gbm = GradientBoostingClassifier(
    n_estimators=80,
    max_depth=4,
    learning_rate=0.1,
    random_state=7,
).fit(X_gbm, y_gbm)

fig_gbm = visualize_decision_geometry(
    model=gbm,
    X=X_gbm,
    y=y_gbm,
    mesh_resolution=28,
    show_confidence=True,
    show_ellipsoids=False,
    title="Gradient Boosting — 4-class Decision Geometry (30-D features)",
)
fig_gbm.show()

---
## 3 · Latent-Space Inspection — High-Dimensional Embeddings (PCA)

`inspect_latent_space` is purpose-built for analysing the geometric structure
of representation spaces: transformer hidden states, GAN latent codes, contrastive
learning embeddings, etc.  Here we simulate 512-dimensional embeddings with
four Gaussian clusters of varying compactness to illustrate how cluster
separation and covariance structure manifest in the projected space.

In [ ]:
rng = np.random.default_rng(0)

# Four clusters with distinct means and anisotropic covariances
cluster_configs = [
    {"mean": np.zeros(512),                  "std": 1.0,  "n": 120},
    {"mean": np.eye(512, 1).ravel() * 6,     "std": 0.7,  "n": 120},
    {"mean": np.eye(512, 2)[:, 1] * 5,       "std": 1.5,  "n": 120},
    {"mean": (np.eye(512, 1) + np.eye(512, 2)).ravel() * 3, "std": 0.9, "n": 120},
]

embeddings = np.vstack([
    rng.normal(loc=c["mean"], scale=c["std"], size=(c["n"], 512))
    for c in cluster_configs
])
emb_labels = np.repeat([0, 1, 2, 3], 120)
emb_names = {0: "Topic: Science", 1: "Topic: Politics", 2: "Topic: Arts", 3: "Topic: Sport"}

fig_latent_pca = inspect_latent_space(
    embeddings=embeddings,
    labels=emb_labels,
    projection_method="pca",
    show_ellipsoids=True,
    show_convex_hulls=True,
    ellipsoid_confidence=0.90,
    class_names=emb_names,
    title="512-D Embeddings — 4 Topic Clusters (PCA Projection)",
)
fig_latent_pca.show()

---
## 4 · Latent-Space Inspection — t-SNE Projection

t-SNE is unsurpassed at revealing local neighbourhood structure in high-
dimensional data.  Note that the t-SNE projection does **not** support
`inverse_transform`, so decision-surface rendering is disabled automatically;
`inspect_latent_space` produces a scatter + ellipsoid scene instead.

In [ ]:
# Use a smaller embedding dimensionality for faster t-SNE
embeddings_64 = embeddings[:, :64].copy()

cfg_tsne = DARK_SCIENTIFIC.with_method("tsne")
cfg_tsne.projection.tsne_perplexity = 30.0
cfg_tsne.projection.tsne_n_iter = 800

fig_tsne = inspect_latent_space(
    embeddings=embeddings_64,
    labels=emb_labels,
    config=cfg_tsne,
    show_ellipsoids=True,
    show_centroids=True,
    class_names=emb_names,
    title="64-D Embeddings — t-SNE Projection",
)
fig_tsne.show()

---
## 5 · Custom Configuration

Every visual property is controlled through `VisualizationConfig`.  The fluent
`.with_*` helpers return modified copies, making it easy to derive a custom
theme without mutating the shared `DARK_SCIENTIFIC` singleton.

In [ ]:
from geolatent import ColorPalette, RenderConfig, ProjectionConfig, VisualizationConfig

# ── Cyberpunk colour palette ──────────────────────────────────────────
cyberpunk_colors = ColorPalette(
    background="#0a0a0f",
    grid="#111122",
    text="#e0e0ff",
    axis_line="#2a2a4a",
    class_colors=[
        "#ff00ff",  # magenta
        "#00ffff",  # cyan
        "#ffff00",  # yellow
        "#ff6600",  # orange
    ],
    annotation_color="#6060aa",
)

custom_cfg = VisualizationConfig(
    colors=cyberpunk_colors,
    render=RenderConfig(
        width=1000,
        height=750,
        surface_opacity=0.35,
        scatter_opacity=0.9,
        marker_size=4,
        show_colorbar=False,
    ),
    projection=ProjectionConfig(
        method="pca",
        scale_input=True,
    ),
)

X_rf, y_rf = make_classification(
    n_samples=350, n_features=15, n_classes=3, n_informative=8, random_state=99
)
rf = RandomForestClassifier(n_estimators=100, random_state=99).fit(X_rf, y_rf)

fig_custom = visualize_decision_geometry(
    model=rf,
    X=X_rf,
    y=y_rf,
    config=custom_cfg,
    title="Random Forest — Custom Cyberpunk Theme",
    show_ellipsoids=True,
)
fig_custom.show()

---
## 6 · MLP Decision Geometry with Convex Hulls

Multi-layer perceptrons learn highly curved, non-linear decision boundaries.
Enabling `show_convex_hulls=True` overlays transparent bounding polyhedra
per class, making the spatial extent of each cluster immediately apparent.

In [ ]:
X_mlp, y_mlp = make_classification(
    n_samples=500,
    n_features=25,
    n_classes=4,
    n_informative=12,
    n_redundant=5,
    random_state=13,
)

mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation="relu",
    max_iter=300,
    random_state=13,
).fit(X_mlp, y_mlp)

fig_mlp = visualize_decision_geometry(
    model=mlp,
    X=X_mlp,
    y=y_mlp,
    mesh_resolution=25,
    show_confidence=True,
    show_ellipsoids=False,
    show_convex_hulls=True,
    title="MLP (128→64→32) Decision Geometry — 4-class, 25-D features",
    class_names={0: "Alpha", 1: "Beta", 2: "Gamma", 3: "Delta"},
)
fig_mlp.show()

---
## 7 · Advanced: Lower-Level Pipeline API

The high-level functions delegate to modular building blocks that can be
composed directly for custom research pipelines.  Here we demonstrate
manual scene construction with a trajectory overlay — useful for visualising
gradient-descent paths or attention walks.

In [ ]:
from geolatent.core.projector import DimensionalityProjector
from geolatent.core.mesh_builder import MeshBuilder
from geolatent.core.geometry import GeometryUtils
from geolatent.rendering.scene import Scene3D
from geolatent.rendering.surfaces import DecisionSurfaceRenderer
from geolatent.rendering.overlays import DataOverlay

X_adv, y_adv = make_blobs(
    n_samples=300, centers=3, n_features=10, cluster_std=1.2, random_state=5
)
clf_adv = SVC(kernel="rbf", probability=True, C=5.0).fit(X_adv, y_adv)

cfg = DARK_SCIENTIFIC.copy()
cfg.projection.method = "pca"

projector = DimensionalityProjector(cfg.projection)
result = projector.fit_transform(X_adv)
X_3d = result.coordinates

mesh = MeshBuilder(resolution=28).build_prediction_mesh(clf_adv, projector, X_3d)

# Simulate an optimisation trajectory in projected space
rng = np.random.default_rng(99)
t = np.linspace(0, 4 * np.pi, 60)
traj_x = 2.5 * np.cos(t) + rng.normal(0, 0.1, 60)
traj_y = 2.5 * np.sin(t) + rng.normal(0, 0.1, 60)
traj_z = np.linspace(X_3d[:, 2].min(), X_3d[:, 2].max(), 60)
waypoints = np.column_stack([traj_x, traj_y, traj_z])

scene = Scene3D(cfg)
scene.set_axis_labels(result.axis_labels)
scene.set_title("Custom Pipeline — Decision Surfaces + Optimisation Trajectory")

surface_renderer = DecisionSurfaceRenderer(cfg)
overlay = DataOverlay(cfg)

scene.add_traces(surface_renderer.render(mesh, show_confidence=True))
scene.add_traces(overlay.render_scatter(X_3d, y_adv))
scene.add_trace(overlay.render_centroids(X_3d, y_adv))
scene.add_traces(overlay.render_ellipsoids(X_3d, y_adv, confidence=0.80))
scene.add_traces(overlay.render_trajectory(waypoints, name="Optimisation path", color="#f0f6fc"))
scene.add_variance_annotation(result.explained_variance_ratio)

fig_advanced = scene.render()
fig_advanced.show()

---
## 8 · Static Export

Figures can be exported to PNG or SVG for publication using `kaleido`.

In [ ]:
# Install kaleido for static export (run once)
# !pip install -q kaleido

# Export the SVM figure as a high-resolution PNG
# fig_svm.write_image("svm_decision_geometry.png", scale=2.0)

# Export as interactive HTML (no kaleido required)
# fig_svm.write_html("svm_decision_geometry.html", include_plotlyjs='cdn')

print("Export cell ready — uncomment the relevant line to save.")

---

## Summary of demonstrated API

| Function / Class | Purpose |
|---|---|
| `visualize_decision_geometry(model, X, y)` | 3-D model decision surfaces |
| `inspect_latent_space(embeddings, labels)` | Embedding manifold visualisation |
| `VisualizationConfig` | Master configuration container |
| `DARK_SCIENTIFIC` | Default dark-scientific theme |
| `DimensionalityProjector` | PCA / t-SNE / UMAP pipeline |
| `MeshBuilder` | Prediction-mesh construction |
| `Scene3D` | Dark-themed Plotly scene manager |
| `DecisionSurfaceRenderer` | Isosurface / volume trace generator |
| `DataOverlay` | Scatter · centroids · ellipsoids · trajectories |

For full API documentation, see the module docstrings or the project README.